# Digital Twin - Deployable Website

Using the principles from the exploratory, preliminary build - I will now build a deployable version of the digital twin that will be hosted on a website.

In [ ]:
# Imports

from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr

In [ ]:
load_dotenv(override=True)
openai = OpenAI()

In [ ]:
# Pushover Setup

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

In [ ]:
# Quick push test to see if it works

def push(message):
    print(f"Push: {message}")
    payload = {'user': pushover_user, 'token': pushover_token, 'message': message}
    requests.post(pushover_url, data=payload)

In [ ]:
push("Hello from the Digital Twin Lab!")

In [ ]:
# Function that pushes notification to Pushover with user details

def record_user_details(email, name='Name not provided', notes=None):
    push(f"Recording interst from {name} wiht email {email} and notes notes {notes}")
    return "OK"

### Quality inspection for improvement

When users send in data and questions, reviewing the interactions allow for spotting areas of improvrement:
- Considerations for additional information provision to the system
- Finding points of weakness or inefficiency in the solution

In [ ]:
# Pushing questions that couldn't be answered due to lack of information to Pushover for inspection

def record_unknown_question(question):
    push(f"{question} was asked and I couldn't answer")
    return "OK"

## Tool-setting

In [ ]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provide an email address",
    "parameters": {
        "type":"object",
        "properties": {
            "email": {"type":"string", "description": "The user's email address"},
            "name": {"type":"string", "description": "The user's name if it was provided"},
            "notes": {"type":"string", "description": "Any additional info about the conversation that's worth recording to give context"}
        },
        "required":["email"],
        "additionalProperties": False
    }
}

In [ ]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this too to record any question that couldn't be answered as you didn't know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {"type":"string", "description": "the question that coud not be answered"},
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [ ]:
tools = [{"type":"function", "function": record_user_details_json},
        {"type":"function", "function": record_unknown_question_json}]

In [ ]:
tools

In [ ]:
# Handling the tool calls with a function

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else "No tool found"
        results.append({'role': 'tool', 'content': json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [ ]:
# Extracting personal and professional information from documents

reader = PdfReader("career_summary_cv_coverletter_linkedin.pdf")
career = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        career += text

with open("summary.txt", "r", encoding="utf-8") as file:
    summary = file.read()

In [ ]:
print(career)

In [ ]:
# Constructing a system prompt

system_prompt = f"""

# Your role

You are a digital twin runing on a webisite, chatting with visitors of the website.
You represent the person whose website you are on.
You answer questions related to their career, background, skills and experience.

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a combination of the person's Linkedin profile, CV and a cover letter example  so that you can answer questions:

{career}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Only answer questions that are related to the user's career, background, skills and experence; steer the conversation
back to professional topics if a question deviates from this or points you away from this.

Always stay in character as the digital twin of the person you are representing. Represent the person.

IMPORTANT: If you do not know the answer to a question, say so. Never make up an answer. 
If the user asks about something not in the context, say you don't know.
"""

In [ ]:
def chat(message,history):
    messages = [{"role": "system", "content":system_prompt}] + history + [{"role":"user", "content":message}]
    response = openai.chat.completions.create(
        model='gpt-6-sol',
        messages=messages,
        tools=tools,
        reasoning_effort='none'
        )
    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = openai.chat.completions.create(
            model='gpt-6-sol',
            messages=messages,
            tools=tools,
            reasoning_effort='none'
            )
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)